In [29]:
import pandas as pd
user_file = pd.read_csv("../generated_data/user_profiles_fixed.csv")

user_file


,Unnamed: 0,gpt_id,id,created_at.users,description,location,protected,verified_type,username,public_metrics.followers_count,...,gpt_speculation_QUOTA_PARTISAN LOYALTIES,"gpt_speculation_QUOTA_IDEOLOGICALLY, THIS PERSON APPEARS TO BE",gpt_speculation_QUOTA_PAST VOTE - TURNOUT IN THE 2020 PRESIDENTIAL ELECTION,gpt_speculation_QUOTA_PAST VOTE - TURNOUT IN THE 2022 HOUSE OF REPRESENTATIVES ELECTION,gpt_speculation_QUOTA_PAST VOTE - VOTE CHOICE IN THE 2020 PRESIDENTIAL ELECTION,gpt_speculation_QUOTA_PAST VOTE - VOTE CHOICE IN THE 2022 HOUSE OF REPRESENTATIVES ELECTION,clean...pool,clean...pool.date,clean...state_simple,image_accessible
0,0,00c19e12d0a58a60a3755015586487d328dd28b9762b1c...,1081532502966239232,2019-01-05T12:46:57.000Z,"i think a lot, listen to music. i’m fond of fl...",they/he ☆ 20,False,none,xxaleb,4426,...,60,50,0.0,60,60,60,trad,2024-09-29T10:51:31Z,USA,True
1,3054,00c8f3ca2e8d1641c2577d2043db8c3d689444afb3f28d...,1542629834953719810,2022-06-30T22:03:26.000Z,A mom of 5 with common sense! Hates being lie...,"Ohio, USA",False,none,Badash1821,625,...,35,25,25.0,35,30,40,radical,2024-10-15T05:18:27Z,Ohio,True
2,3308,01c1c7bf06253f1a936d7c14b1b12c7ea63c4d5965367b...,2865078826,2014-11-07T03:10:21.000Z,Shall not be infringed. Guns. Bourbon. Tiny wr...,At The Range,False,blue,Bossy_Leah,89892,...,20,20,20.0,30,30,30,trad,2024-09-19T15:18:26Z,USA,True
3,5150,0278afc18b853f9d91187b7aa846d2e3a356ab5bb5654b...,1449812808,2013-05-22T20:30:48.000Z,it’s just twitter | 🇵🇦 | Philippians 4:13 #Fly...,"Everywhere, USA",False,none,Dee11___,821,...,85,70,80.0,80,90,95,discourse,2024-09-19T15:19:42Z,USA,False
4,8317,028b411c9fc3afdc4afb03ddc4c340d61903fd5b1c7e6d...,3327892595,2015-06-15T20:32:36.000Z,Emerg doc & researcher. Born @CO2 340ppm. View...,Falling by the wayside,False,none,NashvegasEM,1189,...,60,40,40.0,40,50,50,health,2024-10-15T05:31:55Z,USA,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
923,2139693,fdac85bf55eaec8c287322f213ea35709044be70115bc7...,1544118721261850631,2022-07-05T00:39:47.000Z,"""""I cannot tell a lie"""" - George Washington\n""...",Lizard Lick NC,False,none,KevinInCarolina,420,...,20,20,20.0,20,30,50,democracy,2024-10-28T04:54:54Z,North Carolina,True
924,2141413,fdff322c150ffc45ba1c315a0910fa0b48b3e7bc709612...,1395081597238583303,2021-05-19T18:19:30.000Z,Christian. Husband. Father. Army dad. Charis B...,"Divide, Colorado",False,none,mark_petereit,1196,...,10,10,20.0,20,20,20,democracy,2024-09-24T16:20:27Z,Colorado,True
925,2144564,fe446ad4178e9118f4c1b3db6556f2be127b4e8d88302f...,90123499,2009-11-15T08:09:19.000Z,Dulsura por mi Mami de San Juan Puerto Rico Fu...,USA,False,none,dulcebombomcito,1420,...,65,75,80.0,80,85,85,crime,2024-10-15T05:17:33Z,USA,True
926,2144648,ff533d344660aab4ce616c5420059d225c41d849930117...,1669335694420918272,2023-06-15T13:27:30.000Z,"Running for State Rep District 36, in Grandvie...","Grandview, MO",False,blue,ElectDaveThomas,175,...,10,10,20.0,20,20,30,trad,2024-09-19T15:18:26Z,Missouri,True


In [ ]:
import os
import requests
import tempfile
import base64
import mimetypes
import shutil

def send_prompt(
    system=None,
    prompt=None,
    image_url=None,             # if set, we download to a temp file and inline it
    image_path=None,            # local file path (sent as data URL)
    image_detail="auto",        
    seed=None,
    azure_api_key=os.getenv("AZURE_API_KEY"),
    azure_chat_url=None,        # built if not provided
    azure_model_name=os.getenv("AZURE_MODEL_NAME"),
    max_completion_tokens= 1024,
    temperature=float(os.getenv("TEMPERATURE", "1.0")),
    top_p=float(os.getenv("TOP_P", "1.0")),
    frequency_penalty=float(os.getenv("FREQUENCY_PENALTY", "0.0")),
    presence_penalty=float(os.getenv("PRESENCE_PENALTY", "0.0")),
    max_time=int(os.getenv("MAX_TIME", "30"))
):

    # --- default azure_chat_url if not supplied ---
    if azure_chat_url is None:
        azure_endpoint_base = "https://aipolling.services.ai.azure.com/models"
        azure_api_version  = "2024-05-01-preview"
        azure_chat_url = f"{azure_endpoint_base}/chat/completions?api-version={azure_api_version}"

    # --- helpers ---
    def guess_mime_from_ext(path):
        """Guess MIME type from file extension, fallback to image/jpeg."""
        mime, _ = mimetypes.guess_type(path)
        return mime if mime else "image/jpeg"

    def image_to_data_url(path, mime=None):
        """Read file, encode as base64, return as data URL."""
        if not os.path.exists(path):
            raise FileNotFoundError(f"image_path not found: {path}")
        if mime is None:
            mime = guess_mime_from_ext(path)
        with open(path, "rb") as f:
            encoded = base64.b64encode(f.read()).decode("utf-8")
        return f"data:{mime};base64,{encoded}"

    tmp_path = None
    # If image_url is provided, download it
    if image_url:
        ext = os.path.splitext(image_url)[1].lower()
        suffix = ext if ext else ".bin"
        tmp_fd, tmp_path = tempfile.mkstemp(suffix=suffix)
        os.close(tmp_fd)  # we’ll reopen with write
        resp_img = requests.get(image_url, timeout=max_time)
        if resp_img.status_code >= 400:
            raise RuntimeError(f"Failed to download image_url (HTTP {resp_img.status_code}): {image_url}")
        with open(tmp_path, "wb") as f:
            f.write(resp_img.content)
        image_path = tmp_path

    # --- build messages ---
    messages_list = []
    if system:
        messages_list.append({"role": "system", "content": system})

    user_content = [{"type": "text", "text": prompt}]

    if image_path:
        mime = guess_mime_from_ext(image_path)
        data_url = image_to_data_url(image_path, mime)
        user_content.append({
            "type": "image_url",
            "image_url": {"url": data_url, "detail": image_detail}
        })

    messages_list.append({"role": "user", "content": user_content})

    # --- body payload ---
    body = {
        "model": azure_model_name,
        "messages": messages_list,
        "temperature": float(temperature),
        "top_p": float(top_p),
        "frequency_penalty": float(frequency_penalty),
        "presence_penalty": float(presence_penalty),
        "max_completion_tokens": int(max_completion_tokens),
        "max_tokens": int(max_completion_tokens),  # compat
        "seed": int(seed) if seed is not None else None
    }

    # --- request ---
    headers = {
        "Content-Type": "application/json",
        "Accept": "application/json",
        "api-key": azure_api_key
    }

    resp = requests.post(
        azure_chat_url,
        headers=headers,
        json=body,
        timeout=max_time
    )

    raw_content = resp.text
    if resp.status_code != 200:
        raise RuntimeError(f"API request failed with status {resp.status_code}, body: {raw_content}")

    result = resp.json()

    # --- cleanup tmp file if used ---
    if tmp_path and os.path.exists(tmp_path):
        try:
            os.remove(tmp_path)
        except Exception:
            pass

    return result


In [28]:
res = send_prompt(
    prompt = """
This is a social media user's profile image.

1. List all visible physical characteristics of any people (skin tone, hairstyle, facial hair, wrinkles, clothing, etc.).

2. List all visible contextual or lifestyle cues in the image (e.g., objects, backgrounds, locations, clothing style, accessories, logos, symbols).

3. Based only on those visible cues, summarize possible demographic information only if it can be inferred with reasonable certainty.:
   - Approximate age range
   - Gender presentation
   - Racial or ethnic background
   - Education level (only if directly indicated, e.g., graduation attire)
   - Income level (only if strongly suggested by visible lifestyle cues)
   - Marital status (only if directly visible, e.g., wedding ring or couple photo)
   - Partisan or political affiliation (only if explicit logos, signs, or symbols are visible)

If uncertain or not visible, explicitly state that the information is unclear.

4. Speculation Score: For each demographic attribute, provide a speculation score using the following scale:

0–20 (Low speculation): The image provides clear and direct information relevant to the attribute (e.g., explicit visual evidence in the profile image).

21–40 (Moderate-low speculation): The image provides indirect but strong indicators relevant to the attribute.

41–60 (Moderate speculation): The image provides some hints or partial information relevant to the attribute.

61–80 (Moderate-high speculation): The image provides limited and weak indicators relevant to the attribute.

81–100 (High speculation): The image provides no or almost no information relevant to the attribute.

5. Explanation: For each speculation score, briefly explain why you assigned that score, referring to visible cues or lack thereof.

""",
    image_url="https://pbs.twimg.com/profile_images/776047601816829953/suLLGY-y_400x400.jpg",
    azure_model_name="Llama-4-Maverick-17B-128E-Instruct-FP8"
)

print(res["choices"][0]["message"]["content"])


### 1. Visible Physical Characteristics:
- The person has a fair or light skin tone.
- The person is balding with short brown hair on the sides.
- The person has a short beard and mustache.
- There are visible wrinkles on the forehead.
- The person is wearing a maroon-colored shirt.

### 2. Visible Contextual or Lifestyle Cues:
- The background includes a whiteboard with handwritten notes, suggesting an academic or professional setting.
- There are papers pinned to a wooden wall or panel behind the person.
- The person has wires attached to their head, indicating some form of technical or scientific experiment.

### 3. Demographic Information:
- **Approximate Age Range**: The person appears to be an adult, likely between 30-50 years old due to the balding hair and facial wrinkles.
- **Gender Presentation**: The person appears to present as male based on facial hair and hairstyle.
- **Racial or Ethnic Background**: The skin tone and facial features suggest a Caucasian background, but th

In [38]:
import os
import pandas as pd
import time
import json

INPUT_CSV = "../generated_data/user_profiles_fixed.csv"
OUTPUT_CSV = "../generated_data/more_user_profiles_with_imgdesc.csv"
CHECKPOINT_FILE = "checkpoint.json"

def load_checkpoint():
    """Load checkpointed row index if exists."""
    if os.path.exists(CHECKPOINT_FILE):
        with open(CHECKPOINT_FILE, "r") as f:
            return json.load(f).get("last_completed_index", -1)
    return -1

def save_checkpoint(index):
    """Save last successfully processed row index."""
    with open(CHECKPOINT_FILE, "w") as f:
        json.dump({"last_completed_index": index}, f)

def process_profiles():
    df = pd.read_csv(INPUT_CSV)

    # Make sure the column exists even if we resume
    if "profile_image_description" not in df.columns:
        df["profile_image_description"] = None

    last_completed_index = load_checkpoint()
    print(f"Resuming from row {last_completed_index + 1}")

    for idx in range(last_completed_index + 1, len(df)):
        row = df.iloc[idx]
        image_url = row.get("profile_image_url")

        print(f"[{idx+1}/{len(df)}] Processing user_id={row.get('id', idx)}")

        if pd.isna(image_url) or not str(image_url).startswith("http"):
            desc = "Profile image description is not available for this user"
        else:
            try:
                result = send_prompt(
                    prompt="""
This is a social media user's profile image.

1. List all visible physical characteristics of any people (skin tone, hairstyle, facial hair, wrinkles, clothing, etc.).

2. List all visible contextual or lifestyle cues in the image (e.g., objects, backgrounds, locations, clothing style, accessories, logos, symbols).

3. Based only on those visible cues, summarize possible demographic information only if it can be inferred with reasonable certainty.:
   - Approximate age range
   - Gender presentation
   - Racial or ethnic background
   - Education level (only if directly indicated, e.g., graduation attire)
   - Income level (only if strongly suggested by visible lifestyle cues)
   - Marital status (only if directly visible, e.g., wedding ring or couple photo)
   - Partisan or political affiliation (only if explicit logos, signs, or symbols are visible)

If uncertain or not visible, explicitly state that the information is unclear.

4. Speculation Score: For each demographic attribute, provide a speculation score using the following scale:

0–20 (Low speculation): The image provides clear and direct information relevant to the attribute (e.g., explicit visual evidence in the profile image).

21–40 (Moderate-low speculation): The image provides indirect but strong indicators relevant to the attribute.

41–60 (Moderate speculation): The image provides some hints or partial information relevant to the attribute.

61–80 (Moderate-high speculation): The image provides limited and weak indicators relevant to the attribute.

81–100 (High speculation): The image provides no or almost no information relevant to the attribute.

5. Explanation: For each speculation score, briefly explain why you assigned that score, referring to visible cues or lack thereof.
                    """,
                    image_url=image_url,
                    azure_model_name="Llama-4-Maverick-17B-128E-Instruct-FP8"
                )
                desc = result["choices"][0]["message"]["content"]
            except Exception as e:
                print(f"Error processing image: {e}")
                desc = "Profile image description is not available for this user"

        df.at[idx, "profile_image_description"] = desc

        # Save partial results every row 
        df.to_csv(OUTPUT_CSV, index=False)
        save_checkpoint(idx)
        time.sleep(1)  # avoid API rate limits

    print("Processing complete.")
    # Remove checkpoint when done
    if os.path.exists(CHECKPOINT_FILE):
        os.remove(CHECKPOINT_FILE)

if __name__ == "__main__":
    process_profiles()


Resuming from row 0
[1/928] Processing user_id=1081532502966239232
[2/928] Processing user_id=1542629834953719810
[3/928] Processing user_id=2865078826


KeyboardInterrupt: 

In [31]:
del checkpoint.json

NameError: name 'checkpoint' is not defined

In [44]:
merged.to_csv("../generated_data/user_profiles_with_imgdesc.csv")

In [ ]:
#save the description without specualtion score related info into a seperated column "profile_image_main_info"
import pandas as pd
import re

INPUT_CSV = "../generated_data/user_profiles_with_imgdesc.csv"
OUTPUT_CSV = "../generated_data/final_user_profiles_cleaned.csv"

def remove_sections_from_4_onward(text):
    """
    Keep everything before section 4/Step 4/Speculation Score/Explanation.
    Remove all text starting from section 4.
    """
    if pd.isna(text):
        return None
    
    text = str(text).replace("\r", "")
    
    # Look for headings indicating section 4 or 5
    match = re.search(
        r"(?:###?\s*)?(?:Step\s*4|4\.)[^\n]{0,50}?(speculation|explanation)|speculation\s+score|explanation",
        text,
        flags=re.S | re.I
    )
    if match:
        return text[:match.start()].strip()
    
    # If no section 4 found, keep full text
    return text.strip()

def process_file():
    df = pd.read_csv(INPUT_CSV)

    df["profile_image_main_info"] = df["profile_image_description"].apply(remove_sections_from_4_onward)

    df.to_csv(OUTPUT_CSV, index=False)
    print(f"Processed file saved to {OUTPUT_CSV}")

if __name__ == "__main__":
    process_file()


Processed file saved to ../generated_data/final_user_profiles_cleaned.csv
